In [ ]:
from google.adk.agents import SequentialAgent, LlmAgent

# 第一个代理生成初始草案。
generator = LlmAgent(
    name="DraftWriter",
    description="Generates initial draft content on a given subject.",
    instruction="Write a short, informative paragraph about the user's subject.",
    output_key="draft_text" # 输出保存到该状态键。
)

# 第二个代理人批评第一个代理人的草案。
reviewer = LlmAgent(
    name="FactChecker",
    description="Reviews a given text for factual accuracy and provides a structured critique.",
    instruction="""
    You are a meticulous fact-checker.
    1. Read the text provided in the state key 'draft_text'.
    2. Carefully verify the factual accuracy of all claims.
    3. Your final output must be a dictionary containing two keys:
       - "status": A string, either "ACCURATE" or "INACCURATE".
       - "reasoning": A string providing a clear explanation for your status, citing specific issues if any are found.
    """,
    output_key="review_output" # 结构化字典保存在这里。
)

# SequentialAgent 确保生成器在审阅者之前运行。
review_pipeline = SequentialAgent(
    name="WriteAndReview_Pipeline",
    sub_agents=[generator, reviewer]
)

# 执行流程：
# 1. 生成器运行 -> 将其段落保存到 state['draft_text']。
# 2. 审阅者运行 -> 读取 state['draft_text'] 并将其字典输出保存到 state['review_output']。